In [ ]:
# Install Unsloth and other required libraries
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Maximum context window
dtype = None # Auto-detect
load_in_4bit = True # Squeezes the model to fit on the free T4 GPU

model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"
# model_name = "unsloth/gemma-2-9b-it-bnb-4bit"

print(f"Loading Base Chef: {model_name}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Model loaded successfully!")

In [ ]:
# 1. Attach the LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # The rank/size of the adapter (16 is perfect for this)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
)

print("LoRA Adapters successfully attached! The Chef is ready to learn.")

In [ ]:
from datasets import load_dataset

# Load your custom JSONL file
dataset = load_dataset("json", data_files="/culinary_philosophy_dataset.jsonl", split="train")
# Define the Prompt Template
# This maps your JSON to a conversation format
prompt_template = """Below is an instruction that describes a culinary task. Write a response that appropriately completes the request, applying Vietnamese Five Elements and French technique.

### Instruction:
{}

### Response:
{}"""

# Function to apply the template to every row in your dataset
def format_prompts(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = prompt_template.format(instruction, output)
        texts.append(text)
    return { "text" : texts }

# Process the dataset
dataset = dataset.map(format_prompts, batched = True)

print(f"Dataset formatted! Loaded {len(dataset)} examples.")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        # Set max_steps to roughly the size of your dataset (e.g., 60-100)
        # If you have a small dataset, let it run for 60 steps to learn deeply
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Starting the training process...")
trainer_stats = trainer.train()
print("🎉 Training Complete! You have a custom Master Chef AI.")

In [ ]:
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
import os
import torch
from unsloth import FastLanguageModel
from tqdm import tqdm

# 1. Load Model
model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"
max_seq_length = 8192 # Increased for larger context chunks

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# 2. Prepare Data
input_path = "raw_master_theory.txt"
output_path = "extracted_notes_local.txt"

with open(input_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Chunking (4000 chars is safe for Llama-3 context window)
chunk_size = 4000
chunks = [raw_text[i:i+chunk_size] for i in range(0, len(raw_text), chunk_size)]
print(f"Total chunks to process: {len(chunks)}")

# 3. Extraction Loop
extracted_notes = []

# System Prompt
system_prompt = (
    "You are a research assistant. Extract rules related to: "
    "1. Five Elements and Flavors. 2. Yin-Yang properties in food. "
    "Ignore medical procedures. Summarize in concise English. "
    "If no data, return 'EMPTY'."
)

for i, chunk in enumerate(tqdm(chunks)):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Extract culinary rules from this text: {chunk}"},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True)
    response = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens = True)[0]

    if "EMPTY" not in response.upper():
        extracted_notes.append(response.strip())

# 4. Save intermediate results
with open(output_path, "w", encoding="utf-8") as f:
    f.write("\n\n--- SECTION ---\n\n".join(extracted_notes))

print(f"Local extraction complete. Notes saved to {output_path}")

In [ ]:
from zhipuai import ZhipuAI

client = ZhipuAI(api_key=os.environ.get("ZAI_API_KEY"))

with open("extracted_notes_local.txt", "r", encoding="utf-8") as f:
    combined_notes = f.read()

reduce_prompt = (
    "Synthesize the following research notes into a professional Knowledge Base. "
    "Format: 1. Five Elements & Flavor Mappings. 2. Yin-Yang (Ohsawa) Properties. 3. Categorization Logic. "
    "Language: English. Output only the technical rules."
)

response = client.chat.completions.create(
    model="glm-4.7-flash",
    messages=[
        {"role": "system", "content": reduce_prompt},
        {"role": "user", "content": combined_notes}
    ]
)

with open("core_philosophy.txt", "w", encoding="utf-8") as f:
    f.write(response.choices[0].message.content)

print("Final core_philosophy.txt created using hybrid local/API approach.")

In [ ]:
import torch
from tqdm import tqdm

# 1. Define specific sub-targets to ensure the model doesn't drift
target_distribution = {
    "Grains, Legumes, and Seeds": 500,
    "Common Garden Vegetables": 600,
    "Roots and Sea Vegetables": 400,
    "Proteins (Sustainable & Common)": 300,
    "Pantry Staples and Ferments": 200
}

all_ingredients = []

# 2. Iterative Generation Loop
print("Starting Turbo Expansion...")

for category, count in target_distribution.items():
    print(f"Expanding Category: {category} (Target: {count})")

    # We generate in chunks of 50 to maintain high quality and prevent repetition
    chunks = (count // 50) + (1 if count % 50 != 0 else 0)

    for i in tqdm(range(chunks)):
        batch_prompt = f"""
        Act as a Global Culinary Scientist.
        List 50 unique, common, and diverse culinary ingredients for the category: {category}.

        CRITICAL RULES:
        - Focus on everyday household ingredients worldwide.
        - Avoid luxury/specialty bias.
        - Do not repeat items already listed.
        - Output ONLY a plain text list, one item per line. No numbers.
        """

        # Tokenize and push to GPU
        inputs = tokenizer.apply_chat_template(
            [{"role": "user", "content": batch_prompt}],
            add_generation_prompt = True,
            return_tensors = "pt"
        ).to("cuda")

        # Use more aggressive sampling to ensure diversity across chunks
        outputs = model.generate(
            input_ids = inputs,
            max_new_tokens = 1024,
            temperature = 0.85,
            top_p = 0.95,
            do_sample = True,
            use_cache = True
        )

        response = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]

        # Clean and append
        items = [line.strip() for line in response.split('\n') if line.strip() and len(line) < 40]
        all_ingredients.extend(items)

# 3. Final Deduplication and Saving
unique_ingredients = sorted(list(set(all_ingredients)))
print(f"Total Unique Ingredients Generated: {len(unique_ingredients)}")

with open("ingredients_2000.txt", "w", encoding="utf-8") as f:
    for item in unique_ingredients:
        f.write(item + "\n")

print("Deduplicated ingredients_2000.txt created.")

In [ ]:
import json
import os
import torch
from tqdm import tqdm

# 1. Setup for Batching
# Standard requirement for batching: pad from the left so the model
# focuses on the new tokens at the end.
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

batch_size = 32
input_file = "ingredients_2000.txt"
output_file = "massive_culinary_dataset.jsonl"

with open(input_file, "r", encoding="utf-8") as f:
    ingredients = [line.strip() for line in f if line.strip()]

# 2. Group ingredients into batches
batches = [ingredients[i:i + batch_size] for i in range(0, len(ingredients), batch_size)]

print(f"Processing {len(ingredients)} items in {len(batches)} batches of {batch_size}...")

for batch in tqdm(batches):
    # Prepare a list of prompts for the entire batch
    prompts = []
    for item in batch:
        prompt = f"""
        Knowledge Base: {philosophy_context[:1000]}
        Task: Generate a JSON entry for: {item}
        Rules: Element/Flavor mapping + Yin-Yang classification + French Technique.
        Format: {{"instruction": "...", "output": "..."}}
        """
        messages = [{"role": "user", "content": prompt}]
        # Apply the chat template to each message in the batch
        formatted = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        prompts.append(formatted)

    # Tokenize the entire batch at once
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")

    # 3. Generate for the whole batch in parallel
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id
        )

    # 4. Decode and save results
    # We only care about the tokens generated AFTER the input prompt
    generated_ids = [output[len(input_id):] for input_id, output in zip(inputs.input_ids, outputs)]
    responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    with open(output_file, "a", encoding="utf-8") as f:
        for response in responses:
            try:
                start = response.find("{")
                end = response.rfind("}") + 1
                clean_json = response[start:end]
                json_data = json.loads(clean_json)
                f.write(json.dumps(json_data, ensure_ascii=False) + "\n")
            except:
                continue

print(f"Turbo Synthesis Complete. Dataset saved to {output_file}")